In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mu and Sigma
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t+1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            w = w / w.sum()
    else:
        w = w / s

    return w


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1)
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying transaction cost schedule
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    """
    Returns cost_series of shape (T,)

    Interpretation:
    - cost_series[t] is the cost coefficient used when moving into w_t

    base_cost examples:
    0.0010 = 10 bps
    0.0020 = 20 bps
    """

    T, n = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    elif mode == "vol_scaled":
        # simple cross-sectional average absolute return as market stress proxy
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t+1])

        # normalise around 1
        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0
        vol_state_norm = vol_state / denom

        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    else:
        raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) One-step optimisation with varying lambda_t
# ============================================================
def solve_one_step_varying_cost(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.01,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    tc_penalty = cp.norm1(w - w_prev)

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, S) - lam_t * tc_penalty)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max, fully_invested=True))

    ok = solve_problem(prob)
    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_one_step_varying_cost(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_varying_cost(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 6) Gross and net realised portfolio returns
# ============================================================
def realised_returns_gross(R, W):
    """
    Uses W[t-1] on R[t]
    """
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    """
    Net returns after subtracting cost of rebalancing.

    At date t:
    - gross return uses W[t-1] on R[t]
    - trading cost for entering W[t-1] is charged using turnover from W[t-2] -> W[t-1]

    So net series has length T-1, aligned with gross returns.
    """
    gross = realised_returns_gross(R, W)

    T = W.shape[0]
    trade_costs = np.zeros(T - 1)

    # for gross return at index k, corresponding market move is R[k+1]
    # weights used are W[k]
    # cost charged is for move into W[k], i.e. from W[k-1] to W[k]
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 7) Main
# ============================================================
if __name__ == "__main__":
    seed = 7
    n_assets = 50
    lookback = 252

    gamma = 5.0
    w_max = 0.10
    ridge = 1e-3
    use_log_returns = False

    # transaction cost settings
    base_cost = 0.0020      # 20 bps
    cost_mode = "vol_scaled"   # "constant" or "vol_scaled"
    vol_lookback = 21
    alpha = 2.0

    # load data
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    # returns and moments
    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=252,
        use_log_returns=use_log_returns
    )

    # varying cost series
    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    # run one-step strategy
    W = run_one_step_varying_cost(
        mu_series=mu_series,
        Sigma_series=Sigma_series,
        cost_series=cost_series,
        gamma=gamma,
        w_max=w_max,
        ridge=ridge
    )

    # turnover
    to = turnover(W)
    print("\nAverage turnover")
    print("----------------")
    print(np.mean(to))

    # gross and net returns
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    # stats
    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    print_stats("Gross stats", stats_gross)
    print_stats("Net stats", stats_net)

    print("\nAverage cost coefficient")
    print("------------------------")
    print(np.mean(cost_series))

    print("\nAverage realised trading cost per period")
    print("----------------------------------------")
    print(np.mean(tc_paid))

    # ========================================================
    # plots
    # ========================================================
    plt.figure(figsize=(10, 4))
    plt.plot(cost_series)
    plt.title("Time-varying cost coefficient")
    plt.xlabel("t")
    plt.ylabel("cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(to)
    plt.title("Turnover per rebalance")
    plt.xlabel("t")
    plt.ylabel("sum |Δw|")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.imshow(W.T, aspect="auto")
    plt.title("One-step weights heatmap")
    plt.xlabel("t")
    plt.ylabel("asset")
    plt.colorbar(label="weight")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(tc_paid)
    plt.title("Realised trading cost paid each period")
    plt.xlabel("t")
    plt.ylabel("cost")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(eq_gross, label="Gross equity")
    plt.plot(eq_net, label="Net equity")
    plt.title("Gross vs net equity")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

we added one more constraint 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mu and Sigma
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    if T <= lookback:
        raise ValueError(f"Not enough history: T={T}, need > {lookback}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t+1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            w = w / w.sum()
    else:
        w = w / s

    return w


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying cost schedule
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    """
    Returns cost_series of shape (T,)

    cost_series[t] is the turnover penalty / cost coefficient
    used when moving into w_t.
    """
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t+1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) Improved one-step optimisation:
#    time-varying turnover penalty
#    + hard turnover cap
#    + L2 trade smoothing
# ============================================================
def solve_one_step_capped_smooth(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    """
    Solves:

    max_w  mu'w - gamma * w'Sigma w - lam_t * ||w - w_prev||_1 - eta * ||w - w_prev||_2^2

    s.t.   sum(w)=1, w>=0, w<=w_max, ||w - w_prev||_1 <= tau
    """
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
        - eta * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_one_step_capped_smooth(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_capped_smooth(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 6) Realised gross and net returns
# ============================================================
def realised_returns_gross(R, W):
    """
    Causal:
    use W[t-1] on R[t]
    """
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    """
    gross[k] uses W[k] on R[k+1]
    trading cost at k is the cost of moving W[k-1] -> W[k]
    """
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 7) Main
# ============================================================
if __name__ == "__main__":
    # -------------------------
    # Parameters
    # -------------------------
    seed = 7
    n_assets = 50
    lookback = 252
    use_log_returns = False

    gamma = 5.0
    eta = 2.0          # L2 smoothing on trades
    tau = 0.05         # hard turnover cap per rebalance
    w_max = 0.10
    ridge = 1e-3

    base_cost = 0.0020     # 20 bps
    cost_mode = "vol_scaled"   # "constant" or "vol_scaled"
    vol_lookback = 21
    alpha = 2.0

    # -------------------------
    # Load data
    # -------------------------
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    # -------------------------
    # Returns and rolling moments
    # -------------------------
    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=252,
        use_log_returns=use_log_returns
    )

    # -------------------------
    # Time-varying costs
    # -------------------------
    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    # -------------------------
    # Run improved strategy
    # -------------------------
    W = run_one_step_capped_smooth(
        mu_series=mu_series,
        Sigma_series=Sigma_series,
        cost_series=cost_series,
        gamma=gamma,
        eta=eta,
        tau=tau,
        w_max=w_max,
        ridge=ridge
    )

    # -------------------------
    # Turnover
    # -------------------------
    to = turnover(W)
    print("\nAverage turnover")
    print("----------------")
    print(np.mean(to))

    print("\nTurnover cap used")
    print("-----------------")
    print(tau)

    # -------------------------
    # Gross and net returns
    # -------------------------
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    print_stats("Gross stats", stats_gross)
    print_stats("Net stats", stats_net)

    print("\nAverage cost coefficient")
    print("------------------------")
    print(np.mean(cost_series))

    print("\nAverage realised trading cost per period")
    print("----------------------------------------")
    print(np.mean(tc_paid))

    print("\nMax realised turnover")
    print("---------------------")
    print(np.max(to) if len(to) > 0 else np.nan)

    # ========================================================
    # plots
    # ========================================================
    plt.figure(figsize=(10, 4))
    plt.plot(cost_series)
    plt.title("Time-varying cost coefficient")
    plt.xlabel("t")
    plt.ylabel("cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(to)
    plt.axhline(tau, linestyle="--", label="turnover cap")
    plt.title("Turnover per rebalance")
    plt.xlabel("t")
    plt.ylabel("sum |Δw|")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.imshow(W.T, aspect="auto")
    plt.title("Weights heatmap")
    plt.xlabel("t")
    plt.ylabel("asset")
    plt.colorbar(label="weight")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(tc_paid)
    plt.title("Realised trading cost paid each period")
    plt.xlabel("t")
    plt.ylabel("cost")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(eq_gross, label="Gross equity")
    plt.plot(eq_net, label="Net equity")
    plt.title("Gross vs net equity")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

simple and ehnaced in one

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mean and covariance
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    if T <= lookback:
        raise ValueError(f"Not enough history: T={T}, need > {lookback}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t+1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            s2 = w.sum()
            if s2 <= 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s2
    else:
        w = w / s

    return w


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying cost series
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t+1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) SIMPLE optimisation
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1
# ============================================================
def solve_one_step_simple(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
    )

    constraints = add_constraints(
        w,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_simple_strategy(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_simple(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 6) ENHANCED optimisation
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1 - eta ||w-w_prev||_2^2
#    s.t. ||w-w_prev||_1 <= tau
# ============================================================
def solve_one_step_enhanced(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
        - eta * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_enhanced_strategy(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_enhanced(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 7) Single interface for both strategies
# ============================================================
def run_strategy(
    strategy,
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    strategy = strategy.lower()

    if strategy == "simple":
        return run_simple_strategy(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            gamma=gamma,
            w_max=w_max,
            ridge=ridge,
            w0=w0
        )

    if strategy == "enhanced":
        return run_enhanced_strategy(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            gamma=gamma,
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge,
            w0=w0
        )

    raise ValueError("strategy must be either 'simple' or 'enhanced'")


# ============================================================
# 8) Realised returns
# ============================================================
def realised_returns_gross(R, W):
    """
    Uses W[t-1] on R[t]
    """
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 9) One experiment for one seed
# ============================================================
def run_single_experiment(
    seed,
    strategy="simple",
    n_assets=50,
    lookback=252,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualise=252
):
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)

    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=annualise,
        use_log_returns=use_log_returns
    )

    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    W = run_strategy(
        strategy=strategy,
        mu_series=mu_series,
        Sigma_series=Sigma_series,
        cost_series=cost_series,
        gamma=gamma,
        eta=eta,
        tau=tau,
        w_max=w_max,
        ridge=ridge
    )

    to = turnover(W)
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    result = {
        "seed": seed,
        "strategy": strategy,
        "chosen_assets": chosen_assets,
        "times": times,
        "R_daily": R_daily,
        "mu_series": mu_series,
        "Sigma_series": Sigma_series,
        "cost_series": cost_series,
        "W": W,
        "turnover": to,
        "rp_gross": rp_gross,
        "rp_net": rp_net,
        "tc_paid": tc_paid,
        "eq_gross": eq_gross,
        "eq_net": eq_net,
        "stats_gross": stats_gross,
        "stats_net": stats_net,
        "avg_turnover": np.mean(to) if len(to) > 0 else np.nan,
        "max_turnover": np.max(to) if len(to) > 0 else np.nan,
        "avg_cost_coeff": np.mean(cost_series),
        "avg_realised_tc": np.mean(tc_paid) if len(tc_paid) > 0 else np.nan,
    }

    return result


# ============================================================
# 10) Run many seeds: 1..25 by default
# ============================================================
def run_many_seeds(
    strategy="simple",
    seeds=range(1, 26),
    n_assets=50,
    lookback=252,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualise=252,
    verbose=True
):
    all_results = []

    for seed in seeds:
        if verbose:
            print(f"Running seed {seed} / {max(seeds) if hasattr(seeds, '__len__') else '...'} | strategy={strategy}")

        res = run_single_experiment(
            seed=seed,
            strategy=strategy,
            n_assets=n_assets,
            lookback=lookback,
            use_log_returns=use_log_returns,
            gamma=gamma,
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge,
            base_cost=base_cost,
            cost_mode=cost_mode,
            vol_lookback=vol_lookback,
            alpha=alpha,
            annualise=annualise
        )
        all_results.append(res)

    summary_rows = []
    for res in all_results:
        row = {
            "seed": res["seed"],
            "strategy": res["strategy"],
            "final_equity_gross": res["eq_gross"][-1] if len(res["eq_gross"]) > 0 else np.nan,
            "final_equity_net": res["eq_net"][-1] if len(res["eq_net"]) > 0 else np.nan,
            "cum_return_gross": res["stats_gross"]["cum_return"],
            "cum_return_net": res["stats_net"]["cum_return"],
            "sharpe_gross": res["stats_gross"]["sharpe"],
            "sharpe_net": res["stats_net"]["sharpe"],
            "max_drawdown_gross": res["stats_gross"]["max_drawdown"],
            "max_drawdown_net": res["stats_net"]["max_drawdown"],
            "avg_turnover": res["avg_turnover"],
            "max_turnover": res["max_turnover"],
            "avg_cost_coeff": res["avg_cost_coeff"],
            "avg_realised_tc": res["avg_realised_tc"],
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    avg_row = summary_df.mean(numeric_only=True).to_dict()
    avg_row["seed"] = "AVERAGE"
    avg_row["strategy"] = strategy

    summary_with_avg = pd.concat(
        [summary_df, pd.DataFrame([avg_row])],
        ignore_index=True
    )

    return all_results, summary_df, summary_with_avg


# ============================================================
# 11) Plot helper
# ============================================================
def plot_experiment(result, show_weights_heatmap=True):
    strategy = result["strategy"]
    seed = result["seed"]

    plt.figure(figsize=(10, 4))
    plt.plot(result["cost_series"])
    plt.title(f"Cost series | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["turnover"], label="turnover")
    if strategy == "enhanced":
        plt.axhline(0.05, linestyle="--", label="tau reference")
    plt.title(f"Turnover | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("sum |Δw|")
    plt.legend()
    plt.tight_layout()
    plt.show()

    if show_weights_heatmap:
        plt.figure(figsize=(10, 4))
        plt.imshow(result["W"].T, aspect="auto")
        plt.title(f"Weights heatmap | {strategy} | seed={seed}")
        plt.xlabel("t")
        plt.ylabel("asset")
        plt.colorbar(label="weight")
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["tc_paid"])
    plt.title(f"Realised trading cost | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("cost")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["eq_gross"], label="Gross equity")
    plt.plot(result["eq_net"], label="Net equity")
    plt.title(f"Gross vs net equity | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 12) EXAMPLES
# ============================================================

# -------------------------
# Example A: run ONE seed
# -------------------------
result_simple = run_single_experiment(seed=1, strategy="simple")
result_enhanced = run_single_experiment(seed=1, strategy="enhanced")
print_stats("Simple net stats", result_simple["stats_net"])
print_stats("Enhanced net stats", result_enhanced["stats_net"])
plot_experiment(result_simple)
plot_experiment(result_enhanced)

run for 25 seeds

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50, data=None):
    if data is None:
        data = qndata.stocks.load_data()

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mean and covariance
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    if T <= lookback:
        raise ValueError(f"Not enough history: T={T}, need > {lookback}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t + 1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            s2 = w.sum()
            if s2 <= 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s2
    else:
        w = w / s

    return w


def turnover_from_weights(W):
    if W.shape[0] < 2:
        return np.array([])
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying cost series
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t + 1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) SIMPLE optimisation
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1
# ============================================================
def solve_one_step_simple(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
    )

    constraints = add_constraints(
        w,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_simple_strategy(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_simple(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 6) ENHANCED optimisation
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1 - eta ||w-w_prev||_2^2
#    s.t. ||w-w_prev||_1 <= tau
# ============================================================
def solve_one_step_enhanced(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
        - eta * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_enhanced_strategy(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_enhanced(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 7) Single interface for both strategies
# ============================================================
def run_strategy(
    strategy,
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    strategy = strategy.lower()

    if strategy == "simple":
        return run_simple_strategy(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            gamma=gamma,
            w_max=w_max,
            ridge=ridge,
            w0=w0
        )

    if strategy == "enhanced":
        return run_enhanced_strategy(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            gamma=gamma,
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge,
            w0=w0
        )

    raise ValueError("strategy must be either 'simple' or 'enhanced'")


# ============================================================
# 8) Realised returns
# ============================================================
def realised_returns_gross(R, W):
    """
    Uses W[t-1] on R[t]
    """
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 9) One experiment for one seed
# ============================================================
def run_single_experiment(
    seed,
    strategy="simple",
    n_assets=50,
    lookback=252,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualise=252,
    data=None
):
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets, data=data)

    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=annualise,
        use_log_returns=use_log_returns
    )

    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    W = run_strategy(
        strategy=strategy,
        mu_series=mu_series,
        Sigma_series=Sigma_series,
        cost_series=cost_series,
        gamma=gamma,
        eta=eta,
        tau=tau,
        w_max=w_max,
        ridge=ridge
    )

    to = turnover_from_weights(W)
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    result = {
        "seed": seed,
        "strategy": strategy,
        "chosen_assets": chosen_assets,
        "times": times,
        "R_daily": R_daily,
        "mu_series": mu_series,
        "Sigma_series": Sigma_series,
        "cost_series": cost_series,
        "W": W,
        "turnover": to,
        "rp_gross": rp_gross,
        "rp_net": rp_net,
        "tc_paid": tc_paid,
        "eq_gross": eq_gross,
        "eq_net": eq_net,
        "stats_gross": stats_gross,
        "stats_net": stats_net,
        "avg_turnover": np.mean(to) if len(to) > 0 else np.nan,
        "max_turnover": np.max(to) if len(to) > 0 else np.nan,
        "avg_max_weight": float(np.mean(np.max(W, axis=1))) if W.size > 0 else np.nan,
        "has_nan_weights": bool(np.isnan(W).any()),
        "avg_cost_coeff": np.mean(cost_series),
        "avg_realised_tc": np.mean(tc_paid) if len(tc_paid) > 0 else np.nan,
    }

    return result


# ============================================================
# 10) CSV save helpers
# ============================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_chosen_assets_csv(seed_dir, seed, chosen_assets):
    df = pd.DataFrame({
        "seed": seed,
        "asset": chosen_assets
    })
    df.to_csv(os.path.join(seed_dir, f"chosen_assets_seed_{seed}.csv"), index=False)
    return df


def save_summary_csv(seed_dir, seed, results):
    rows = []
    for r in results:
        rows.append({
            "seed": seed,
            "strategy": r["strategy"],
            "final_equity_gross": r["eq_gross"][-1] if len(r["eq_gross"]) > 0 else np.nan,
            "final_equity_net": r["eq_net"][-1] if len(r["eq_net"]) > 0 else np.nan,
            "cum_return_gross": r["stats_gross"]["cum_return"],
            "cum_return_net": r["stats_net"]["cum_return"],
            "sharpe_gross": r["stats_gross"]["sharpe"],
            "sharpe_net": r["stats_net"]["sharpe"],
            "max_drawdown_gross": r["stats_gross"]["max_drawdown"],
            "max_drawdown_net": r["stats_net"]["max_drawdown"],
            "avg_turnover": r["avg_turnover"],
            "max_turnover": r["max_turnover"],
            "avg_max_weight": r["avg_max_weight"],
            "has_nan_weights": r["has_nan_weights"],
            "avg_cost_coeff": r["avg_cost_coeff"],
            "avg_realised_tc": r["avg_realised_tc"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"summary_seed_{seed}.csv"), index=False)
    return df


def save_timeseries_csv(seed_dir, seed, times, results):
    rows = []
    time_index = pd.to_datetime(times[1:])

    for r in results:
        for i, dt in enumerate(time_index):
            rows.append({
                "seed": seed,
                "date": dt,
                "strategy": r["strategy"],
                "eq_gross": r["eq_gross"][i],
                "eq_net": r["eq_net"][i],
                "rp_gross": r["rp_gross"][i],
                "rp_net": r["rp_net"][i],
                "turnover": r["turnover"][i] if i < len(r["turnover"]) else np.nan,
                "trading_cost_paid": r["tc_paid"][i] if i < len(r["tc_paid"]) else np.nan,
                "cost_coeff": r["cost_series"][i + 1] if i + 1 < len(r["cost_series"]) else np.nan,
            })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"timeseries_seed_{seed}.csv"), index=False)
    return df


def save_weights_csv(seed_dir, seed, times, chosen_assets, weights_dict):
    weight_rows = []
    time_index = pd.to_datetime(times)

    for strategy_name, W in weights_dict.items():
        for t, dt in enumerate(time_index):
            row = {
                "seed": seed,
                "date": dt,
                "strategy": strategy_name
            }
            for j, asset in enumerate(chosen_assets):
                row[str(asset)] = W[t, j]
            weight_rows.append(row)

    df = pd.DataFrame(weight_rows)
    df.to_csv(os.path.join(seed_dir, f"weights_seed_{seed}.csv"), index=False)
    return df


def save_cost_series_csv(seed_dir, seed, times, results):
    rows = []
    time_index = pd.to_datetime(times)

    for r in results:
        for i, dt in enumerate(time_index):
            rows.append({
                "seed": seed,
                "date": dt,
                "strategy": r["strategy"],
                "cost_coeff": r["cost_series"][i],
            })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"cost_series_seed_{seed}.csv"), index=False)
    return df


def save_sanity_csv(seed_dir, seed, results):
    row = {"seed": seed}

    for r in results:
        name = r["strategy"].lower()
        row[f"nan_{name}"] = bool(np.isnan(r["W"]).any())
        row[f"avg_max_weight_{name}"] = float(np.mean(r["W"].max(axis=1)))
        row[f"avg_turnover_{name}"] = float(np.mean(turnover_from_weights(r["W"]))) if r["W"].shape[0] > 1 else np.nan

    sanity = pd.DataFrame([row])
    sanity.to_csv(os.path.join(seed_dir, f"sanity_seed_{seed}.csv"), index=False)
    return sanity


def save_failed_seed_csv(seed_dir, seed, error_message):
    df = pd.DataFrame([{
        "seed": seed,
        "error": str(error_message)
    }])
    df.to_csv(os.path.join(seed_dir, f"failed_seed_{seed}.csv"), index=False)
    return df


# ============================================================
# 11) Run one seed and save everything
# ============================================================
def run_single_seed_and_save(
    seed,
    data,
    out_dir,
    config
):
    seed_dir = os.path.join(out_dir, "seeds", f"seed_{seed}")
    ensure_dir(seed_dir)

    _, chosen_assets = load_quantiacs_universe(
        seed=seed,
        n_assets=config["n_assets"],
        data=data
    )
    save_chosen_assets_csv(seed_dir, seed, chosen_assets)

    print(f"\nRunning seed {seed} | first 10 assets: {chosen_assets[:10]}")

    result_simple = run_single_experiment(
        seed=seed,
        strategy="simple",
        n_assets=config["n_assets"],
        lookback=config["lookback"],
        use_log_returns=config["use_log_returns"],
        gamma=config["gamma"],
        eta=config["eta"],
        tau=config["tau"],
        w_max=config["w_max"],
        ridge=config["ridge"],
        base_cost=config["base_cost"],
        cost_mode=config["cost_mode"],
        vol_lookback=config["vol_lookback"],
        alpha=config["alpha"],
        annualise=config["annualise"],
        data=data
    )

    result_enhanced = run_single_experiment(
        seed=seed,
        strategy="enhanced",
        n_assets=config["n_assets"],
        lookback=config["lookback"],
        use_log_returns=config["use_log_returns"],
        gamma=config["gamma"],
        eta=config["eta"],
        tau=config["tau"],
        w_max=config["w_max"],
        ridge=config["ridge"],
        base_cost=config["base_cost"],
        cost_mode=config["cost_mode"],
        vol_lookback=config["vol_lookback"],
        alpha=config["alpha"],
        annualise=config["annualise"],
        data=data
    )

    results = [result_simple, result_enhanced]
    times = result_simple["times"]

    save_summary_csv(seed_dir, seed, results)
    save_timeseries_csv(seed_dir, seed, times, results)
    save_cost_series_csv(seed_dir, seed, times, results)
    save_sanity_csv(seed_dir, seed, results)

    if config["save_weights"]:
        weights_dict = {
            "simple": result_simple["W"],
            "enhanced": result_enhanced["W"],
        }
        save_weights_csv(seed_dir, seed, times, chosen_assets, weights_dict)

    return {
        "seed": seed,
        "chosen_assets": chosen_assets,
        "results": results,
    }


# ============================================================
# 12) Run many seeds and save combined CSVs
# ============================================================
def run_many_seeds_and_save(
    seeds=range(1, 26),
    out_dir="cost",
    config=None,
    verbose=True
):
    if config is None:
        config = {
            "n_assets": 50,
            "lookback": 252,
            "use_log_returns": False,
            "gamma": 5.0,
            "eta": 2.0,
            "tau": 0.05,
            "w_max": 0.10,
            "ridge": 1e-3,
            "base_cost": 0.0020,
            "cost_mode": "vol_scaled",
            "vol_lookback": 21,
            "alpha": 2.0,
            "annualise": 252,
            "save_weights": True,
        }

    ensure_dir(out_dir)
    ensure_dir(os.path.join(out_dir, "seeds"))

    print("Loading Quantiacs data once...")
    data = qndata.stocks.load_data()

    all_summary = []
    all_timeseries = []
    failed_rows = []

    seeds_list = list(seeds)

    for i, seed in enumerate(seeds_list, start=1):
        try:
            if verbose:
                print(f"Running seed {seed} ({i}/{len(seeds_list)})")

            run_out = run_single_seed_and_save(
                seed=seed,
                data=data,
                out_dir=out_dir,
                config=config
            )

            seed_dir = os.path.join(out_dir, "seeds", f"seed_{seed}")
            seed_summary = pd.read_csv(os.path.join(seed_dir, f"summary_seed_{seed}.csv"))
            seed_timeseries = pd.read_csv(os.path.join(seed_dir, f"timeseries_seed_{seed}.csv"))

            all_summary.append(seed_summary)
            all_timeseries.append(seed_timeseries)

        except Exception as e:
            print(f"Seed {seed} failed: {e}")
            seed_dir = os.path.join(out_dir, "seeds", f"seed_{seed}")
            ensure_dir(seed_dir)
            save_failed_seed_csv(seed_dir, seed, e)
            failed_rows.append({
                "seed": seed,
                "error": str(e)
            })

    combined_summary = None
    combined_timeseries = None
    avg_by_strategy = None
    avg_timeseries = None

    if len(all_summary) > 0:
        combined_summary = pd.concat(all_summary, ignore_index=True)
        combined_summary.to_csv(os.path.join(out_dir, "all_seeds_summary.csv"), index=False)

        avg_by_strategy = (
            combined_summary
            .groupby("strategy", as_index=False)
            .agg({
                "final_equity_gross": "mean",
                "final_equity_net": "mean",
                "cum_return_gross": "mean",
                "cum_return_net": "mean",
                "sharpe_gross": "mean",
                "sharpe_net": "mean",
                "max_drawdown_gross": "mean",
                "max_drawdown_net": "mean",
                "avg_turnover": "mean",
                "max_turnover": "mean",
                "avg_max_weight": "mean",
                "avg_cost_coeff": "mean",
                "avg_realised_tc": "mean",
            })
        )
        avg_by_strategy.to_csv(os.path.join(out_dir, "average_metrics_across_seeds.csv"), index=False)

        print("\n=== Average metrics across seeds ===")
        print(avg_by_strategy)

    if len(all_timeseries) > 0:
        combined_timeseries = pd.concat(all_timeseries, ignore_index=True)
        combined_timeseries.to_csv(os.path.join(out_dir, "all_seeds_timeseries.csv"), index=False)

        avg_timeseries = (
            combined_timeseries
            .groupby(["date", "strategy"], as_index=False)
            .agg({
                "eq_gross": "mean",
                "eq_net": "mean",
                "rp_gross": "mean",
                "rp_net": "mean",
                "turnover": "mean",
                "trading_cost_paid": "mean",
                "cost_coeff": "mean",
            })
        )
        avg_timeseries.to_csv(os.path.join(out_dir, "average_timeseries_across_seeds.csv"), index=False)

    if len(failed_rows) > 0:
        failed_df = pd.DataFrame(failed_rows)
        failed_df.to_csv(os.path.join(out_dir, "failed_seeds.csv"), index=False)

    return combined_summary, combined_timeseries, avg_by_strategy, avg_timeseries


# ============================================================
# 13) Plot helper
# ============================================================
def plot_experiment(result, show_weights_heatmap=True):
    strategy = result["strategy"]
    seed = result["seed"]

    plt.figure(figsize=(10, 4))
    plt.plot(result["cost_series"])
    plt.title(f"Cost series | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["turnover"], label="turnover")
    if strategy == "enhanced":
        plt.axhline(0.05, linestyle="--", label="tau reference")
    plt.title(f"Turnover | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("sum |Δw|")
    plt.legend()
    plt.tight_layout()
    plt.show()

    if show_weights_heatmap:
        plt.figure(figsize=(10, 4))
        plt.imshow(result["W"].T, aspect="auto")
        plt.title(f"Weights heatmap | {strategy} | seed={seed}")
        plt.xlabel("t")
        plt.ylabel("asset")
        plt.colorbar(label="weight")
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["tc_paid"])
    plt.title(f"Realised trading cost | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("cost")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["eq_gross"], label="Gross equity")
    plt.plot(result["eq_net"], label="Net equity")
    plt.title(f"Gross vs net equity | {strategy} | seed={seed}")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 14) Main
# ============================================================
if __name__ == "__main__":
    config = {
        "n_assets": 50,
        "lookback": 252,
        "use_log_returns": False,
        "gamma": 5.0,
        "eta": 2.0,
        "tau": 0.05,
        "w_max": 0.10,
        "ridge": 1e-3,
        "base_cost": 0.0020,
        "cost_mode": "vol_scaled",
        "vol_lookback": 21,
        "alpha": 2.0,
        "annualise": 252,
        "save_weights": True,   # set False if weight files become too large
    }

    out_dir = "cost"
    seeds = range(1, 26)

    combined_summary, combined_timeseries, avg_by_strategy, avg_timeseries = run_many_seeds_and_save(
        seeds=seeds,
        out_dir=out_dir,
        config=config,
        verbose=True
    )

    # Optional plots from averages across seeds
    if avg_timeseries is not None and len(avg_timeseries) > 0:
        avg_timeseries["date"] = pd.to_datetime(avg_timeseries["date"])

        plt.figure(figsize=(10, 5))
        for strategy in avg_timeseries["strategy"].unique():
            tmp = avg_timeseries[avg_timeseries["strategy"] == strategy].sort_values("date")
            plt.plot(tmp["date"], tmp["eq_net"], label=strategy)
        plt.title("Average net equity across seeds")
        plt.xlabel("Date")
        plt.ylabel("Equity")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 5))
        for strategy in avg_timeseries["strategy"].unique():
            tmp = avg_timeseries[avg_timeseries["strategy"] == strategy].sort_values("date")
            plt.plot(tmp["date"], tmp["turnover"], label=strategy)
        plt.title("Average turnover across seeds")
        plt.xlabel("Date")
        plt.ylabel("Turnover")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 5))
        for strategy in avg_timeseries["strategy"].unique():
            tmp = avg_timeseries[avg_timeseries["strategy"] == strategy].sort_values("date")
            plt.plot(tmp["date"], tmp["trading_cost_paid"], label=strategy)
        plt.title("Average realised trading cost across seeds")
        plt.xlabel("Date")
        plt.ylabel("Trading cost")
        plt.legend()
        plt.tight_layout()
        plt.show()

    print(f"\nAll done. Results saved in folder: {out_dir}")
    print(f"Per-seed files are in: {os.path.join(out_dir, 'seeds')}")